In [24]:
# !pip install rasterio
# !pip install pystac-client
# !pip install planetary-computer
# !pip install odc
# !pip install rioxarray
# !pip install odc.stac

# Water Quality Prediction: Benchmark Notebook

## Challenge Overview

<p align="justify">
Welcome to the EY AI & Data Challenge 2026!  
The objective of this challenge is to build a robust <strong> machine learning model </strong>capable of predicting water quality across various river locations in South Africa. In addition to accurate predictions, the model should also identify and emphasize the key factors that significantly influence water quality.
</p>

<p align="justify">
Participants will be provided with a dataset containing three water quality parameters <strong>Total Alkalinity</strong>, <strong>Electrical Conductance</strong>, and <strong>Dissolved Reactive Phosphorus</strong> collected between 2011 and 2015 from approximately 200 river locations across South Africa. Each data point includes the geographic coordinates (latitude and longitude) of the sampling site, the date of collection, and the corresponding water quality measurements.
</p>

<p align="justify">
Using this dataset, participants are expected to build a machine learning model to predict water quality parameters for a separate validation dataset, which includes locations from different regions not present in the training data. The challenge also encourages participants to explore feature importance and provide insights into the factors most strongly associated with variations in water quality.
</p>

<p align="justify">
This challenge is designed for participants with varying levels of experience in data science, remote sensing, and environmental analytics. It offers a valuable opportunity to apply machine learning techniques to real-world environmental data and contribute to advancing water quality monitoring using artificial intelligence.
</p>


<b>About the Notebook: </b><p align="justify"> <p>

<p align="justify"> In this notebook, we demonstrate a basic workflow that serves as a foundation for the challenge. The model has been developed to predict <b>water quality parameters</b> using features derived from the <b>Landsat</b> and <b>TerraClimate</b> datasets. Specifically, four spectral bands—<b>SWIR22</b> (Shortwave Infrared 2), <b>NIR</b> (Near Infrared), <b>Green</b>, and <b>SWIR16</b> (Shortwave Infrared 1)—were utilized from Landsat, along with derived spectral indices such as <b>NDMI</b> (Normalized Difference Moisture Index) and <b>MNDWI</b> (Modified Normalized Difference Water Index). In addition, the <b>PET</b> (Potential Evapotranspiration) variable was incorporated from the <b>TerraClimate</b> dataset to account for climatic influences on water quality. </p>

<p align="justify"> The dataset spans a five-year period from <b>2011 to 2015</b>. Using <b>API-based data extraction</b> methods, both Landsat and TerraClimate features were retrieved directly from the <a href="https://planetarycomputer.microsoft.com/">Microsoft Planetary Computer</a> portal. These combined spectral, index-based, and climatic features were used as predictors in a regression model to estimate three key water quality parameters: <b>Total Alkalinity (TA)</b>, <b>Electrical Conductance (EC)</b>, and <b>Dissolved Reactive Phosphorus (DRP)</b>.

</p> <p align="justify"> Please note that this notebook serves only as a starting point. Several assumptions were made during the data extraction and model development process, which you may find opportunities to improve upon. Participants are encouraged to explore additional features, enhance preprocessing techniques, or experiment with different regression algorithms to optimize predictive performance. </p>

## Load In Dependencies

To run this demonstration notebook, you will need to have the following packages imported below installed. This may take some time.  

In [25]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = "/content/drive/My Drive/EY 2026 Optimizing Clean Water Supply"
    WATER_QUALITY_FILE = "water_quality_training_like.csv"
except Exception:
    DATA_DIR = "."
    WATER_QUALITY_FILE = "water_quality_training_dataset.csv"
import os

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Multi-dimensional arrays and datasets (e.g., NetCDF, Zarr)
import xarray as xr

# Geospatial raster data handling with CRS support
import rioxarray as rxr

# Raster operations and spatial windowing
import rasterio
from rasterio.windows import Window

# Feature preprocessing and data splitting
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.spatial import cKDTree

# Machine Learning
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load
from pystac.extensions.eo import EOExtension as eo

from datetime import date
from tqdm import tqdm
import os

## Response Variable

<p align="justify">
Before building the model, we first load the <b>water quality training dataset</b>. The curated dataset contains samples collected from various monitoring stations across the study region. Each record includes the geographical coordinates (Latitude and Longitude), the sample collection date, and the corresponding <b>measured values</b> for the three key water quality parameters — <b>Total Alkalinity (TA)</b>, <b>Electrical Conductance (EC)</b>, and <b>Dissolved Reactive Phosphorus (DRP)</b>.
</p>


In [27]:
import pandas as pd
import numpy as np

# =============================
# 0) Paths (edit these)
# =============================
SAMPLES_PATH = "/content/drive/My Drive/EY 2026 Optimizing Clean Water Supply/samples.csv"
METADATA_XLSX_PATH = "/content/drive/My Drive/EY 2026 Optimizing Clean Water Supply/metadata.xlsx"
OUTPUT_PATH = "/content/drive/My Drive/EY 2026 Optimizing Clean Water Supply/water_quality_training_like.csv"

# =============================
# 1) Load samples (long format)
# =============================
samples = pd.read_csv(SAMPLES_PATH, sep=";")
samples["Sample Date"] = pd.to_datetime(samples["Sample Date"], errors="coerce")

# Keep recommended quality rows
samples = samples[samples["Data Quality"].isin(["Excellent", "Good", "Fair"])].copy()

# Handle value flags
def handle_value_flags(row):
    # '<' : below detection limit → Value / 2
    if row["Value Flags"] == "<":
        return row["Value"] / 2
    return row["Value"]

samples["Value_clean"] = samples.apply(handle_value_flags, axis=1)

# =============================
# 2) Pivot to wide format
# =============================
wide_df = samples.pivot_table(
    index=["GEMS Station Number", "Sample Date"],
    columns="Parameter Code",
    values="Value_clean",
    aggfunc="mean"  # if multiple measurements exist for the same day
).reset_index()

wide_df.columns.name = None

# =============================
# 3) Load station metadata (lat/lon)
# =============================
station_meta = pd.read_excel(METADATA_XLSX_PATH, sheet_name="Station_Metadata")

# Keep only columns needed for merge
station_meta = station_meta[["GEMS Station Number", "Latitude", "Longitude"]].copy()

# Make sure numeric
station_meta["Latitude"] = pd.to_numeric(station_meta["Latitude"], errors="coerce")
station_meta["Longitude"] = pd.to_numeric(station_meta["Longitude"], errors="coerce")

# =============================
# 4) Merge lat/lon into wide_df
# =============================
wide_geo = wide_df.merge(
    station_meta,
    on="GEMS Station Number",
    how="left"
)

# Optional: drop rows without coordinates
wide_geo = wide_geo.dropna(subset=["Latitude", "Longitude"]).copy()

# =============================
# 5) Rename parameter columns to match "training-like" names
#    Use Parameter Long Name from metadata (clean trailing characters)
# =============================
param_meta = pd.read_excel(METADATA_XLSX_PATH, sheet_name="Parameter_Metadata")
param_meta = param_meta[["Parameter Code", "Parameter Long Name"]].copy()

# Clean long names (e.g., remove trailing ')' if present)
param_meta["Parameter Long Name"] = (
    param_meta["Parameter Long Name"]
    .astype(str)
    .str.strip()
    .str.replace(r"\)+$", "", regex=True)
)

code_to_longname = dict(zip(param_meta["Parameter Code"], param_meta["Parameter Long Name"]))

# Rename only columns that are parameter codes
fixed_cols = ["GEMS Station Number", "Sample Date", "Latitude", "Longitude"]
param_cols = [c for c in wide_geo.columns if c not in fixed_cols]

rename_map = {c: code_to_longname.get(c, c) for c in param_cols}
wide_geo = wide_geo.rename(columns=rename_map)

# =============================
# 6) Reorder columns to match water_quality_training_dataset style
#    Latitude, Longitude, Sample Date, (parameters...)
# =============================
parameter_cols_final = [c for c in wide_geo.columns if c not in fixed_cols]
parameter_cols_final = sorted(parameter_cols_final)  # optional: keep columns sorted

final_df = wide_geo[["Latitude", "Longitude", "Sample Date"] + parameter_cols_final].copy()

# Optional: if you want Sample Date format like the training dataset (it looked like "MM-DD-YYYY")
# final_df["Sample Date"] = final_df["Sample Date"].dt.strftime("%m-%d-%Y")

# =============================
# 7) Save
# =============================
final_df.to_csv(OUTPUT_PATH, index=False)
print(final_df.head())
print("Saved to:", OUTPUT_PATH)
print("Shape:", final_df.shape)

   Latitude  Longitude Sample Date  Calcium - Dissolved  Chloride - Dissolved  \
0 -30.06186   28.50975  2011-01-19               16.882                 4.565   
1 -30.06186   28.50975  2011-04-06               19.553                 1.505   
2 -30.06186   28.50975  2011-05-11               18.794                 1.163   
3 -30.06186   28.50975  2011-06-29               24.162                 2.159   
4 -30.06186   28.50975  2011-10-26               23.965                 1.557   

   Dissolved Reactive Phosphorus  Electrical Conductance  \
0                          0.013                   158.8   
1                          0.005                   168.6   
2                          0.005                   188.7   
3                          0.005                   205.2   
4                          0.005                   216.3   

   Fluoride - Dissolved  Free Ammonia Nitrogen  Hardness - Magnesium  ...  \
0                 0.108               0.000822                23.575  ...  

In [28]:
water = pd.read_csv(os.path.join(DATA_DIR, WATER_QUALITY_FILE))
water["Sample Date"] = pd.to_datetime(water["Sample Date"], errors="coerce")
print(water.columns)

Index(['Latitude', 'Longitude', 'Sample Date', 'Calcium - Dissolved',
       'Chloride - Dissolved', 'Dissolved Reactive Phosphorus',
       'Electrical Conductance', 'Fluoride - Dissolved',
       'Free Ammonia Nitrogen', 'Hardness - Magnesium', 'Hardness - Total',
       'Magnesium - Dissolved', 'Nitrate and Nitrite', 'Nitrite',
       'Potassium - Dissolved', 'Salinity', 'Si-Dis', 'Sodium - Dissolved',
       'Sodium Adsorption Ratio', 'Sulfate - Dissolved', 'Total Alkalinity',
       'Total Ammonia Nitrogen', 'Total Kjeldahl Nitrogen', 'Total Nitrogen',
       'Total Phosphorus', 'pH'],
      dtype='object')


## Predictor Variables

<p align="justify">
Now that we have our water quality dataset, the next step is to gather the predictor variables from the <b>Landsat</b> and <b>TerraClimate</b> datasets. In this notebook, we demonstrate how to <b>load previously extracted satellite and climate data</b> from separate files, rather than performing the extraction directly, which allows for a smoother and faster experience. Participants can refer to the dedicated extraction notebooks—one for Landsat and another for TerraClimate—to understand how the data was retrieved and processed, and they can also generate their own output CSV files if needed. Using these pre-extracted CSV files, this notebook focuses on loading the predictor features and running the subsequent analysis and model training efficiently.
</p>
<p align="justify">
For more detailed guidance on the original data extraction process, you can review the <a href="https://planetarycomputer.microsoft.com/dataset/landsat-c2-l2#Example-Notebook">Landsat example notebook</a> and the <a href="https://planetarycomputer.microsoft.com/dataset/terraclimate#Example-Notebook">TerraClimate example notebook</a> available on the Planetary Computer portal.
</p>

<p align="justify">We have used selected spectral bands — SWIR22 (Shortwave Infrared 2), NIR (Near Infrared), Green, and SWIR16 (Shortwave Infrared 1) — and computed key spectral indices such as NDMI (Normalized Difference Moisture Index) and MNDWI (Modified Normalized Difference Water Index). These features capture surface moisture, vegetation, and water content characteristics that influence water quality variability. </p> <p align="justify"> In addition to Landsat features, we also incorporated the <b>Potential Evapotranspiration (PET)</b> variable from the <b>TerraClimate</b> dataset, which provides high-resolution global climate data. The PET feature captures the atmospheric demand for moisture, representing climatic conditions such as temperature, humidity, and radiation that influence surface water evaporation and thus affect water quality parameters. </p> <ul> <li>SWIR22 – Sensitive to surface moisture and turbidity variations in water bodies.</li> <li>NIR – Helps in identifying vegetation and suspended matter in water.</li> <li>Green – Useful for detecting water color and surface reflectance changes.</li> <li>SWIR16 – Provides information on surface dryness and sediment concentration.</li> <li>NDMI – Derived from NIR and SWIR16, indicates moisture and vegetation-water interaction.</li> <li>MNDWI – Derived from Green and SWIR22, effective for distinguishing open water areas and reducing built-up noise.</li> <li>PET – Extracted from the TerraClimate dataset, represents the potential evapotranspiration that influences hydrological and water quality dynamics.</li> </ul>

<h4 style="color:rgb(255, 0, 0)"><strong>Tip 1</strong></h4>  
<p align="justify">  
Participants are encouraged to experiment with different combinations of <b>Landsat</b> bands or even include data from other public satellite data sources. By creating mathematical combinations of bands, you can derive various spectral indices that capture surface and environmental characteristics.
</p>


<h3>Loading Pre-Extracted Landsat Data</h3>
<p align="justify">
In this notebook, we <b>load previously extracted Landsat data</b> from CSV files generated in a separate extraction notebook. This approach ensures a smoother and faster workflow, allowing participants to focus on data analysis and model development without waiting for time-consuming data retrieval.
</p>
<p align="justify">
Participants are expected to generate their own data extraction CSV files by running the dedicated Landsat extraction notebook. These CSV files can then be used here to smoothly run this benchmark notebook. Participants can refer to the extraction notebook to understand the API-based process, including how individual bands and indices like <b>NDMI</b> were computed. Using these pre-extracted CSV files simplifies preprocessing and is ideal for large-scale environmental and water quality analysis.
</p>


<h4 style="color:rgb(255, 0, 0)"><strong>Tip 2</strong></h4>
In the data extraction process (performed in the dedicated extraction notebooks), a 100 m focal buffer was applied around each sampling location rather than using a single point. Participants may explore creating different focal buffers around the locations (e.g., 50 m, 150 m, etc.) during extraction. For example, if a 50 m buffer was used for “Band 2”, the extracted CSV values would reflect the average of "Band 2" within 50 meters of each location. This approach can help reduce errors associated with spatial autocorrelation.


In [29]:
landsat_train_features = pd.read_csv(os.path.join(DATA_DIR, "landsat_features_training.csv"))
landsat_train_features.head()

,Latitude,Longitude,Sample Date,nir,green,swir16,swir22,NDMI,MNDWI
0,-28.760833,17.730278,02-01-2011,11190.0,11426.0,7687.5,7645.0,0.185538,0.195595
1,-26.861111,28.884722,03-01-2011,17658.5,9550.0,13746.5,10574.0,0.124566,-0.180134
2,-26.450000,28.085833,03-01-2011,15210.0,10720.0,17974.0,14201.0,-0.083293,-0.252805
3,-27.671111,27.236944,03-01-2011,14887.0,10943.0,13522.0,11403.0,0.048048,-0.105416
4,-27.356667,27.286389,03-01-2011,16828.5,9502.5,12665.5,9643.0,0.141147,-0.142683


<h3>Loading Pre-Extracted TerraClimate Data</h3>
<p align="justify">
In this notebook, we <b>load previously extracted TerraClimate data</b> from CSV files generated in a dedicated extraction notebook. This approach ensures a smoother and faster workflow, allowing participants to focus on data analysis and model development without waiting for time-consuming data retrieval.
</p>
<p align="justify">
Participants are expected to generate their own data extraction CSV files by running the dedicated TerraClimate extraction notebook. These CSV files can then be used here to smoothly run this benchmark notebook. Participants can refer to the extraction notebook to understand the API-based process, including how climate variables such as <b>Potential Evapotranspiration (PET)</b> were extracted. Using these pre-extracted CSV files ensures consistent, automated retrieval of high-resolution climate data that can be easily integrated with satellite-derived features for comprehensive environmental and hydrological analysis.
</p>


In [30]:
Terraclimate_df = pd.read_csv(os.path.join(DATA_DIR, "terraclimate_features_training.csv"))
Terraclimate_df.head()

,Latitude,Longitude,Sample Date,pet
0,-28.760833,17.730278,02-01-2011,174.2
1,-26.861111,28.884722,03-01-2011,124.1
2,-26.450000,28.085833,03-01-2011,127.5
3,-27.671111,27.236944,03-01-2011,129.7
4,-27.356667,27.286389,03-01-2011,129.2


## Joining the predictor variables and response variables
Now that we have extracted our predictor variables, we need to join them onto the response variable . We use the function <i><b>combine_two_datasets</b></i> to combine the predictor variables and response variables.The <i><b>concat</b></i> function from pandas comes in handy here.

In [31]:
# Combine two datasets vertically (along columns) using pandas concat function.
def combine_two_datasets(dataset1,dataset2,dataset3):
    '''
    Returns a  vertically concatenated dataset.
    Attributes:
    dataset1 - Dataset 1 to be combined
    dataset2 - Dataset 2 to be combined
    '''

    data = pd.concat([dataset1,dataset2,dataset3], axis=1)
    data = data.loc[:, ~data.columns.duplicated()]
    return data

In [32]:
# Combining ground data and final data into a single dataset.
wq_data = combine_two_datasets(water, landsat_train_features, Terraclimate_df)
wq_data.head()

,Latitude,Longitude,Sample Date,Calcium - Dissolved,Chloride - Dissolved,Dissolved Reactive Phosphorus,Electrical Conductance,Fluoride - Dissolved,Free Ammonia Nitrogen,Hardness - Magnesium,...,Total Nitrogen,Total Phosphorus,pH,nir,green,swir16,swir22,NDMI,MNDWI,pet
0,-30.06186,28.50975,2011-01-19,16.882,4.565,0.013,158.8,0.108,0.000822,23.575,...,NaN,NaN,7.618,11190.0,11426.0,7687.5,7645.0,0.185538,0.195595,174.2
1,-30.06186,28.50975,2011-04-06,19.553,1.505,0.005,168.6,0.116,0.000822,24.654,...,NaN,NaN,7.853,17658.5,9550.0,13746.5,10574.0,0.124566,-0.180134,124.1
2,-30.06186,28.50975,2011-05-11,18.794,1.163,0.005,188.7,0.331,0.001645,37.167,...,NaN,NaN,8.176,15210.0,10720.0,17974.0,14201.0,-0.083293,-0.252805,127.5
3,-30.06186,28.50975,2011-06-29,24.162,2.159,0.005,205.2,0.314,0.000822,34.492,...,NaN,NaN,7.993,14887.0,10943.0,13522.0,11403.0,0.048048,-0.105416,129.7
4,-30.06186,28.50975,2011-10-26,23.965,1.557,0.005,216.3,0.421,0.004935,28.616,...,NaN,NaN,8.306,16828.5,9502.5,12665.5,9643.0,0.141147,-0.142683,129.2


In [33]:
wq_data.columns

Index(['Latitude', 'Longitude', 'Sample Date', 'Calcium - Dissolved',
       'Chloride - Dissolved', 'Dissolved Reactive Phosphorus',
       'Electrical Conductance', 'Fluoride - Dissolved',
       'Free Ammonia Nitrogen', 'Hardness - Magnesium', 'Hardness - Total',
       'Magnesium - Dissolved', 'Nitrate and Nitrite', 'Nitrite',
       'Potassium - Dissolved', 'Salinity', 'Si-Dis', 'Sodium - Dissolved',
       'Sodium Adsorption Ratio', 'Sulfate - Dissolved', 'Total Alkalinity',
       'Total Ammonia Nitrogen', 'Total Kjeldahl Nitrogen', 'Total Nitrogen',
       'Total Phosphorus', 'pH', 'nir', 'green', 'swir16', 'swir22', 'NDMI',
       'MNDWI', 'pet'],
      dtype='object')

In [34]:
"""
Feature engineering for water quality prediction (tabular dataset).
No external datasets or APIs. Input: DataFrame with Latitude, Longitude, Sample Date, swir22, NDMI, MNDWI, pet.
"""

from __future__ import annotations

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from typing import Optional, Tuple

try:
    from sklearn.ensemble import RandomForestRegressor
except ImportError:
    RandomForestRegressor = None

# Default numerical columns used for lags and spatial means
NUMERIC_FEATURES = ["swir22", "NDMI", "MNDWI", "pet"]

# Columns that are available at prediction time (Landsat + TerraClimate + engineered from them).
# Use this to keep feature count low and avoid leakage from other water quality parameters.
ML_FEATURE_NAMES = [
    "Latitude", "Longitude",
    "nir", "green", "swir16", "swir22", "NDMI", "MNDWI", "pet",
    "swir22_lag1", "swir22_lag3", "NDMI_lag1", "NDMI_lag3",
    "MNDWI_lag1", "MNDWI_lag3", "pet_lag1", "pet_lag3",
    "swir22_spatial_mean", "NDMI_spatial_mean", "MNDWI_spatial_mean", "pet_spatial_mean",
    "month", "day_of_year", "month_sin", "month_cos", "doy_sin", "doy_cos",
    "NDMI_x_MNDWI", "swir22_x_NDMI", "pet_x_NDMI",
]


def _ensure_datetime(df: pd.DataFrame, date_col: str = "Sample Date") -> pd.DataFrame:
    """Ensure Sample Date is datetime; convert to days since epoch for consistency."""
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    return df


def add_lag_features(
    df: pd.DataFrame,
    group_cols: list[str],
    date_col: str,
    value_cols: list[str],
    lags: list[int],
) -> pd.DataFrame:
    """
    Add lag features by (Latitude, Longitude), sorted by date. No future information.
    """
    df = df.sort_values(group_cols + [date_col]).reset_index(drop=True)
    out = df.copy()
    for col in value_cols:
        for lag in lags:
            out[f"{col}_lag{lag}"] = (
                out.groupby(group_cols)[col].shift(lag).values
            )
    return out


def add_spatial_neighborhood_features(
    df: pd.DataFrame,
    lat_col: str = "Latitude",
    lon_col: str = "Longitude",
    date_col: str = "Sample Date",
    value_cols: list[str] = None,
    radius_deg: Optional[float] = 0.25,
    k_nearest: Optional[int] = None,
    min_neighbors: int = 1,
) -> pd.DataFrame:
    """
    For each row, compute mean of each value_col on the same date:
    - within radius_deg (if radius_deg is set), or
    - among k_nearest neighbors (if k_nearest is set).
    Excludes self when computing mean. No future information.
    """
    if value_cols is None:
        value_cols = [c for c in NUMERIC_FEATURES if c in df.columns]
    out = df.copy()
    for col in value_cols:
        out[f"{col}_spatial_mean"] = np.nan

    dates = out[date_col].dropna().unique()
    for d in dates:
        mask = (out[date_col] == d).values
        n_sub = mask.sum()
        if n_sub < 2:
            continue
        global_inds = np.where(mask)[0]
        pts = out.loc[mask, [lat_col, lon_col]].values.astype(np.float64)
        tree = cKDTree(pts)
        for pos, i in enumerate(global_inds):
            lat, lon = pts[pos]
            if k_nearest is not None:
                # k+1 to include self, then drop self
                k = min(k_nearest + 1, n_sub)
                dists, idx = tree.query([lat, lon], k=k)
                idx = np.atleast_1d(idx)
                neighbors_in_subset = [j for j in idx if j != pos]
            else:
                # query_ball_point: single point can return a flat list, not list-of-lists
                res = tree.query_ball_point([lat, lon], r=radius_deg)
                neighbors_in_subset = np.atleast_1d(res).flatten().tolist()
                neighbors_in_subset = [j for j in neighbors_in_subset if j != pos]
            if len(neighbors_in_subset) < min_neighbors:
                continue
            neighbors_global = global_inds[neighbors_in_subset]
            for col in value_cols:
                if col not in out.columns:
                    continue
                vals = out.iloc[neighbors_global][col].values
                vals = vals[np.isfinite(vals)]
                if len(vals) > 0:
                    out.iloc[i, out.columns.get_loc(f"{col}_spatial_mean")] = np.mean(
                        vals
                    )
    return out


def add_seasonal_features(
    df: pd.DataFrame,
    date_col: str = "Sample Date",
) -> pd.DataFrame:
    """Add month, day_of_year, and cyclic sin/cos encodings."""
    out = df.copy()
    dt = pd.to_datetime(out[date_col], errors="coerce")
    out["month"] = dt.dt.month
    out["day_of_year"] = dt.dt.dayofyear
    # Cyclic: period 12 for month, 365 for doy
    out["month_sin"] = np.sin(2 * np.pi * out["month"] / 12)
    out["month_cos"] = np.cos(2 * np.pi * out["month"] / 12)
    out["doy_sin"] = np.sin(2 * np.pi * out["day_of_year"] / 365.25)
    out["doy_cos"] = np.cos(2 * np.pi * out["day_of_year"] / 365.25)
    return out


def add_regional_dummies(
    df: pd.DataFrame,
    lat_col: str = "Latitude",
    lon_col: str = "Longitude",
    grid_deg: float = 1.0,
    prefix: str = "region",
) -> pd.DataFrame:
    """Bin Lat/Lon into grid_deg x grid_deg and create dummy variables."""
    out = df.copy()
    lat_bin = (out[lat_col] // grid_deg).astype(int).astype(str)
    lon_bin = (out[lon_col] // grid_deg).astype(int).astype(str)
    out["_grid_id"] = lat_bin + "_" + lon_bin
    dummies = pd.get_dummies(out["_grid_id"], prefix=prefix, dtype=float)
    out = pd.concat([out.drop(columns=["_grid_id"]), dummies], axis=1)
    return out


def add_interaction_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add NDMI*MNDWI, swir22*NDMI, pet*NDMI."""
    out = df.copy()
    if "NDMI" in out.columns and "MNDWI" in out.columns:
        out["NDMI_x_MNDWI"] = out["NDMI"] * out["MNDWI"]
    if "swir22" in out.columns and "NDMI" in out.columns:
        out["swir22_x_NDMI"] = out["swir22"] * out["NDMI"]
    if "pet" in out.columns and "NDMI" in out.columns:
        out["pet_x_NDMI"] = out["pet"] * out["NDMI"]
    return out


def fill_missing_safe(df: pd.DataFrame, strategy: str = "median") -> pd.DataFrame:
    """Fill numeric NaN with column median (or mean). Non-numeric columns unchanged."""
    out = df.copy()
    num_cols = out.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        if out[col].isna().any():
            if strategy == "median":
                out[col] = out[col].fillna(out[col].median())
            else:
                out[col] = out[col].fillna(out[col].mean())
    return out


def engineer_features(
    df: pd.DataFrame,
    lat_col: str = "Latitude",
    lon_col: str = "Longitude",
    date_col: str = "Sample Date",
    numeric_cols: Optional[list[str]] = None,
    lags: Optional[list[int]] = None,
    spatial_radius_deg: float = 0.25,
    spatial_k_nearest: Optional[int] = None,
    grid_deg: float = 1.0,
    fill_missing: bool = True,
    add_spatial: bool = True,
    add_regional: bool = True,
) -> pd.DataFrame:
    """
    Full feature engineering pipeline. No future information; safe for ML.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain: Latitude, Longitude, Sample Date, swir22, NDMI, MNDWI, pet.
    numeric_cols : list, optional
        Columns to use for lags and spatial means. Default: ['swir22','NDMI','MNDWI','pet'].
    lags : list, optional
        Lag steps. Default: [1, 3].
    spatial_radius_deg : float
        Radius in degrees for same-date neighborhood mean (used if spatial_k_nearest is None).
    spatial_k_nearest : int, optional
        If set, use mean over k nearest same-date points instead of radius.
    grid_deg : float
        Grid size in degrees for regional dummies (e.g. 1.0 = 1°×1°).
    fill_missing : bool
        If True, fill numeric NaNs with column median.
    add_spatial : bool
        If True, add spatial neighborhood means (can be slow on large data).
    add_regional : bool
        If True, add regional dummy variables.

    Returns
    -------
    pd.DataFrame
        DataFrame with original columns plus engineered features, ready for ML.
    """
    if numeric_cols is None:
        numeric_cols = [c for c in NUMERIC_FEATURES if c in df.columns]
    if lags is None:
        lags = [1, 3]

    out = _ensure_datetime(df, date_col)
    group_cols = [lat_col, lon_col]

    # 1) Lag features (grouped by location, no future info)
    out = add_lag_features(
        out,
        group_cols=group_cols,
        date_col=date_col,
        value_cols=numeric_cols,
        lags=lags,
    )

    # 2) Spatial neighborhood features (same date only; radius or k_nearest)
    if add_spatial:
        out = add_spatial_neighborhood_features(
            out,
            lat_col=lat_col,
            lon_col=lon_col,
            date_col=date_col,
            value_cols=numeric_cols,
            radius_deg=spatial_radius_deg if spatial_k_nearest is None else None,
            k_nearest=spatial_k_nearest,
        )

    # 3) Seasonal features
    out = add_seasonal_features(out, date_col=date_col)

    # 4) Regional dummies (1°×1° grid)
    if add_regional:
        out = add_regional_dummies(
            out, lat_col=lat_col, lon_col=lon_col, grid_deg=grid_deg
        )

    # 5) Interaction features
    out = add_interaction_features(out)

    if fill_missing:
        out = fill_missing_safe(out, strategy="median")

    return out


def select_ml_features(df: pd.DataFrame, exclude: Optional[list[str]] = None) -> pd.DataFrame:
    """
    Return only columns suitable for ML (available at prediction time + engineered).
    Drops regional dummies and other water-quality-only columns to reduce dimension.
    """
    allowed = set(ML_FEATURE_NAMES)
    if exclude:
        allowed -= set(exclude)
    cols = [c for c in df.columns if c in allowed]
    return df[cols].copy()


def drop_low_importance_features(
    X: pd.DataFrame,
    y: pd.Series,
    *,
    model: Optional["RandomForestRegressor"] = None,
    threshold_percentile: Optional[float] = None,
    top_k: Optional[int] = None,
    min_importance: Optional[float] = None,
    random_state: int = 42,
    n_estimators: int = 100,
) -> Tuple[pd.DataFrame, pd.Series, pd.Series]:
    """
    Drop features with low importance from a fitted or newly fitted Random Forest.

    Parameters
    ----------
    X : pd.DataFrame
        Feature matrix (column names must be set).
    y : pd.Series
        Target vector.
    model : RandomForestRegressor, optional
        Pre-fitted model with feature_importances_. If None, fits one on (X, y).
    threshold_percentile : float, optional
        Drop features below this percentile of importance (e.g. 10 = drop bottom 10%).
    top_k : int, optional
        Keep only the top k most important features.
    min_importance : float, optional
        Drop features with importance strictly below this value.
    random_state : int
        For fitting when model is None.
    n_estimators : int
        For fitting when model is None.

    Returns
    -------
    X_selected : pd.DataFrame
        X with low-importance columns dropped.
    y : pd.Series
        Unchanged y (same index as X_selected).
    importances : pd.Series
        Feature importances (index = feature name).
    """
    if RandomForestRegressor is None:
        raise ImportError("sklearn is required for drop_low_importance_features")

    if model is None:
        X_clean = X.dropna(axis=1, how="all").fillna(X.median())
        mask = y.notna()
        X_clean = X_clean.loc[mask]
        y_clean = y.loc[mask]
        model = RandomForestRegressor(
            n_estimators=n_estimators, random_state=random_state
        )
        model.fit(X_clean, y_clean)
        feat_names = X_clean.columns
    else:
        X_clean = X
        y_clean = y
        feat_names = X.columns

    imp = pd.Series(model.feature_importances_, index=feat_names).sort_values(
        ascending=False
    )

    if top_k is not None:
        keep = imp.head(top_k).index.tolist()
    elif threshold_percentile is not None:
        th = np.nanpercentile(imp.values, threshold_percentile)
        keep = imp.index[imp >= th].tolist()
    elif min_importance is not None:
        keep = imp.index[imp >= min_importance].tolist()
    else:
        # default: drop bottom 25% by percentile
        th = np.nanpercentile(imp.values, 25)
        keep = imp.index[imp >= th].tolist()

    if not keep:
        keep = [imp.index[0]]

    X_selected = X_clean[keep].copy()
    return X_selected, y_clean, imp


# ---------------------------------------------------------------------------
# Training X/y and submission features (for benchmark notebook flow)
# ---------------------------------------------------------------------------

TARGET_NAMES = [
    "Total Alkalinity",
    "Electrical Conductance",
    "Dissolved Reactive Phosphorus",
]


def prepare_training_Xy(
    wq_data: pd.DataFrame,
    target_names: Optional[list[str]] = None,
    use_importance_drop: bool = False,
    importance_threshold_percentile: Optional[float] = 25,
    importance_top_k: Optional[int] = None,
    random_state: int = 42,
) -> Tuple[pd.DataFrame, dict, list, Optional[dict]]:
    """
    Build X and y_TA, y_EC, y_DRP from wq_data using ML_FEATURE_NAMES.
    Optionally drop low-importance features (per target or using first target).

    Parameters
    ----------
    wq_data : pd.DataFrame
        Engineered training data (e.g. after engineer_features).
    target_names : list, optional
        Default: Total Alkalinity, Electrical Conductance, Dissolved Reactive Phosphorus.
    use_importance_drop : bool
        If True, run drop_low_importance_features for the first target and use
        the same feature set for all three targets.
    importance_threshold_percentile : float, optional
        Passed to drop_low_importance_features when use_importance_drop=True.
    importance_top_k : int, optional
        If set, keep only top_k features instead of threshold_percentile.
    random_state : int
        For drop_low_importance_features.

    Returns
    -------
    X : pd.DataFrame
        Feature matrix (same columns for all targets).
    y_dict : dict
        {target_name: pd.Series} for each target.
    feature_cols : list
        Column names used in X (use for submission_val_data).
    importances_dict : dict or None
        If use_importance_drop, {target_name: pd.Series of importances}; else None.
    """
    if target_names is None:
        target_names = list(TARGET_NAMES)
    feature_cols = [c for c in ML_FEATURE_NAMES if c in wq_data.columns]
    X_full = wq_data[feature_cols].copy()
    y_dict = {t: wq_data[t] for t in target_names if t in wq_data.columns}
    importances_dict = None

    if use_importance_drop and y_dict:
        first_target = next(iter(y_dict))
        y_first = y_dict[first_target]
        kwargs = {"random_state": random_state}
        if importance_top_k is not None:
            kwargs["top_k"] = importance_top_k
        else:
            kwargs["threshold_percentile"] = importance_threshold_percentile or 25
        X_selected, y_aligned, imp = drop_low_importance_features(
            X_full, y_first, **kwargs
        )
        feature_cols = X_selected.columns.tolist()
        X_full = X_selected
        importances_dict = {first_target: imp}
        y_dict[first_target] = y_aligned
        for t in y_dict:
            if t != first_target:
                y_dict[t] = y_dict[t].reindex(X_full.index)

    # Drop rows where any target is NaN
    mask = pd.Series(True, index=X_full.index)
    for t in y_dict:
        mask = mask & y_dict[t].notna()
    for t in y_dict:
        y_dict[t] = y_dict[t].loc[mask]
    X_full = X_full.loc[mask]
    return X_full, y_dict, feature_cols, importances_dict


def prepare_submission_features(
    val_data: pd.DataFrame,
    feature_cols: list[str],
    fill_missing: float = 0.0,
) -> pd.DataFrame:
    """
    Build submission feature matrix from val_data with the same columns as training.

    Parameters
    ----------
    val_data : pd.DataFrame
        Engineered validation data (same pipeline as wq_data).
    feature_cols : list
        List of column names from prepare_training_Xy (or ML_FEATURE_NAMES subset).
    fill_missing : float
        Value for missing columns or NaN (default 0).

    Returns
    -------
    pd.DataFrame
        val_data reindexed to feature_cols, ready for scaler.transform / model.predict.
    """
    out = val_data.reindex(columns=feature_cols, fill_value=fill_missing)
    out = out.fillna(fill_missing)
    return out

In [35]:
# Ensure Sample Date is datetime for engineer_features (if it was stored as days)
if pd.api.types.is_numeric_dtype(wq_data["Sample Date"]):
    wq_data["Sample Date"] = pd.to_datetime(wq_data["Sample Date"], unit="D", origin="1970-01-01")
wq_data = engineer_features(
    wq_data,
    spatial_radius_deg=0.25,
    add_spatial=True,
    add_regional=False,
    fill_missing=True,
)

<h3>Handling Missing Values</h3>  
<p align="justify">  
Before model training, missing values in the dataset were carefully handled to ensure data consistency and prevent model bias. Numerical columns were imputed using their median values, maintaining the overall data distribution while minimizing the impact of outliers.  
</p>

In [36]:
wq_data = wq_data.fillna(wq_data.median(numeric_only=True))
wq_data.isna().sum()

,0
Latitude,0
Longitude,0
Sample Date,0
Calcium - Dissolved,0
Chloride - Dissolved,0
Dissolved Reactive Phosphorus,0
Electrical Conductance,0
Fluoride - Dissolved,0
Free Ammonia Nitrogen,0
Hardness - Magnesium,0


## Model Building

<p align="justify"> Now let us select the columns required for our model building exercise. We will consider only Band swir22, NDMI and MNDWI from the Landsat data and pet from Terraclimate dataset as our predictor variables. It does not make sense to use latitude and longitude as predictor variables, as they do not have any direct impact on predicting the water quality parameters.</p>


In [37]:
# Retaining only the columns for swir22, NDMI, MNDWI, pet, Total Alkalinity, Electrical Conductance and Dissolved Reactive Phosphorus Index in the dataset.
wq_data = wq_data[['Latitude', 'Longitude', 'Sample Date', 'Calcium - Dissolved',
       'Chloride - Dissolved', 'Dissolved Reactive Phosphorus',
       'Electrical Conductance', 'Fluoride - Dissolved',
       'Free Ammonia Nitrogen', 'Hardness - Magnesium', 'Hardness - Total',
       'Magnesium - Dissolved', 'Nitrate and Nitrite', 'Nitrite',
       'Potassium - Dissolved', 'Salinity', 'Si-Dis', 'Sodium - Dissolved',
       'Sodium Adsorption Ratio', 'Sulfate - Dissolved', 'Total Alkalinity',
       'Total Ammonia Nitrogen', 'Total Kjeldahl Nitrogen', 'Total Nitrogen',
       'Total Phosphorus', 'pH', 'nir', 'green', 'swir16', 'swir22', 'NDMI',
       'MNDWI', 'pet']]

<h4 style="color:rgb(255, 0, 0)"><strong>Tip 3</strong></h4>
<p align="justify">We are developing individual models for each water quality parameter using a common set of features: SWIR22, NDMI, MNDWI, and PET. However, participants are encouraged to experiment with different feature combinations to build more robust machine learning models.</p>

## Helper Functions
### Train and Test Split
<p align="justify">We will now split the data into 70% training data and 30% test data. Scikit-learn alias “sklearn” is a robust library for machine learning in Python. The scikit-learn library has a <i><b>model_selection</b></i> module in which there is a splitting function <i><b>train_test_split</b></i>. You can use the same.</p>

### Feature Scaling

<p align="justify"> Before initiating the model training we may have to execute different data pre-processing steps. Here we are demonstrating the scaling of 'swir22','NDMI','MNDWI','pet' variable by using Standard Scaler.</p>

<p align = "justify">Feature Scaling is a data preprocessing step for numerical features. Many machine learning algorithms like Gradient descent methods, KNN algorithm, linear and logistic regression, etc. require data scaling to produce good results. Scikit learn provides functions that can be used to apply data scaling. Here we are using Standard Scaler. The idea behind Standard Scaler is that it will transform your data such that its distribution will have a mean value 0 and standard deviation of 1.</p>

### Model Training
<p align="justify">
Now that we have the data in a format suitable for machine learning, we can begin training our models. In this demonstration notebook, we will build three separate regression models — one for each target water quality parameter: Total Alkalinity, Electrical Conductance, and Dissolved Reactive Phosphorus.
Each model will be trained independently to capture the unique relationships between the satellite-derived features and each parameter.
</p>

<p align="justify">
We will use the Random Forest Regressor from the scikit-learn library to build our models. Scikit-learn provides a wide range of regression algorithms with extensive parameter tuning and customization capabilities.
</p>

<p align="justify">
For model training, the predictor variables (e.g., SWIR22, NDMI, MNDWI, and pet) will be stored in an array X, and the response variable (one of the three water quality parameters) will be stored in an array Y.
It is important not to include the response variable in X. Additionally, since latitude, longitude, and sample date are only used for spatial and temporal reference, they will be excluded from the predictor variables during model training.
</p>

### Model Evaluation
<p align="justify">
Now that we have trained our models for the three water quality parameters, the next step is to evaluate their performance. Each regression model for Total Alkalinity, Electrical Conductance, and Dissolved Reactive Phosphorus is assessed using the R² score and the Root Mean Square Error (RMSE). The R² score measures how well the model explains the variance in the observed values, while RMSE quantifies the average magnitude of prediction errors. Together, these metrics help determine how effectively each model captures variations in water quality across different locations and sampling dates. Scikit-learn provides built-in functions to compute these metrics, and participants may explore additional evaluation methods or custom metrics as needed.</p>


<h4 style="color:rgb(255, 0, 0)"><strong>Tip 4</strong></h4>
<p align="justify">There are many data preprocessing methods available, which might help to improve the model performance. Participants should explore various suitable preprocessing methods as well as different machine learning algorithms to build a robust model.</p>

In [38]:
def split_data(X, y, test_size=0.3, random_state=42):
    return train_test_split(X, y, test_size=test_size, random_state=random_state)

def scale_data(X_train, X_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, scaler

def train_model(X_train_scaled, y_train):
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train_scaled, y_train)
    return model

def evaluate_model(model, X_scaled, y_true, dataset_name="Test"):
    y_pred = model.predict(X_scaled)
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"\n{dataset_name} Evaluation:")
    print(f"R²: {r2:.3f}")
    print(f"RMSE: {rmse:.3f}")
    return y_pred, r2, rmse

<div class="section">
  <h2>Model Workflow (Pipeline)</h2>
  <p align="justify">
    The complete model development process follows a structured pipeline to ensure consistency, reproducibility, and clarity.
    Each stage in the workflow is modularized into independent functions, which can be reused for different water quality parameters.
    This modular approach helps streamline the process and makes the workflow easily adaptable to new datasets or parameters in the future.
  </p>

  <p align="justify">
    The pipeline automates the sequence of steps — from data preparation to evaluation — for each target parameter.
    The same set of predictor variables is used, while the response variable changes for each of the three targets:
    <i>Total Alkalinity (TA)</i>, <i>Electrical Conductance (EC)</i>, and <i>Dissolved Reactive Phosphorus (DRP)</i>.
    By maintaining a consistent framework, comparisons across models remain fair and interpretable.
  </p>


In [39]:
def run_pipeline(X, y, param_name="Parameter"):
    print(f"\n{'='*60}")
    print(f"Training Model for {param_name}")
    print(f"{'='*60}")

    # Split data
    X_train, X_test, y_train, y_test = split_data(X, y)

    # Scale
    X_train_scaled, X_test_scaled, scaler = scale_data(X_train, X_test)

    # Train
    model = train_model(X_train_scaled, y_train)

    # Evaluate (in-sample)
    y_train_pred, r2_train, rmse_train = evaluate_model(model, X_train_scaled, y_train, "Train")

    # Evaluate (out-sample)
    y_test_pred, r2_test, rmse_test = evaluate_model(model, X_test_scaled, y_test, "Test")

    # Return summary
    results = {
        "Parameter": param_name,
        "R2_Train": r2_train,
        "RMSE_Train": rmse_train,
        "R2_Test": r2_test,
        "RMSE_Test": rmse_test
    }
    return model, scaler, pd.DataFrame([results])

### Model Training and Evaluation for Each Parameter

<p align="justify">In this step, we apply the complete modeling pipeline to each of the three selected water quality parameters — Total Alkalinity, Electrical Conductance, and Dissolved Reactive Phosphorus. The input feature set (<code>X</code>) remains the same across all three models, while the target variable (<code>y</code>) changes for each parameter. For every parameter, the <code>run_pipeline()</code> function is executed, which handles data preprocessing, model training, and both in-sample and out-of-sample evaluation. This ensures a consistent workflow and allows for a fair comparison of model performance across different water quality indicators.</p>

In [40]:
import numpy as np
import pandas as pd

def add_time_features(df, date_col="Sample Date"):
    d = pd.to_datetime(df[date_col], errors="coerce")
    df["month"] = d.dt.month
    df["dayofyear"] = d.dt.dayofyear
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    return df

def add_spatial_poly(df):
    df["lat_lon"] = df["Latitude"] * df["Longitude"]
    df["lat2"] = df["Latitude"] ** 2
    df["lon2"] = df["Longitude"] ** 2
    return df

def add_interactions(df):
    df["NDMI_x_MNDWI"] = df["NDMI"] * df["MNDWI"]
    df["swir22_x_NDMI"] = df["swir22"] * df["NDMI"]
    df["pet_x_NDMI"] = df["pet"] * df["NDMI"]
    # safe ratio (avoid div by zero)
    df["MNDWI_div_NDMI"] = df["MNDWI"] / (df["NDMI"].replace(0, np.nan))
    return df

def add_lag_and_rolling(df, group_cols=("Latitude", "Longitude"), date_col="Sample Date",
                        cols=("swir22", "NDMI", "MNDWI", "pet"),
                        lags=(1, 3), rolling_windows=(3, 7)):
    df = df.sort_values(list(group_cols) + [date_col]).copy()
    g = df.groupby(list(group_cols), sort=False)
    # Lag features
    for c in cols:
        for L in lags:
            df[f"{c}_lag{L}"] = g[c].shift(L)
    # Rolling features (past only: shift(1) then roll within group)
    for c in cols:
        for w in rolling_windows:
            df[f"{c}_rollmean{w}"] = g[c].transform(lambda x: x.shift(1).rolling(w, min_periods=1).mean())
            df[f"{c}_rollstd{w}"] = g[c].transform(lambda x: x.shift(1).rolling(w, min_periods=1).std())
    return df

def add_spatial_means_same_date(df, date_col="Sample Date",
                                cols=("swir22", "NDMI", "MNDWI", "pet"),
                                grid_deg=0.25):
    """Approximate spatial neighborhood mean via lat/lon grid bins (same date)."""
    df = df.copy()
    df["lat_bin"] = (df["Latitude"] / grid_deg).round().astype("Int64")
    df["lon_bin"] = (df["Longitude"] / grid_deg).round().astype("Int64")
    group_keys = ["lat_bin", "lon_bin", date_col]
    grp = df.groupby(group_keys, sort=False)
    for c in cols:
        df[f"{c}_spmean_grid"] = grp[c].transform("mean")
    df = df.drop(columns=["lat_bin", "lon_bin"])
    return df

def build_features(landsat_df, terraclimate_df, template_df,
                  use_extra_raw=True,
                  use_spatial_grid_mean=True,
                  use_lag_rolling=True):
    """Build ML-ready feature matrix for BOTH training and submission. Call the same way for train/val."""
    X = pd.DataFrame({
        "swir22": landsat_df["swir22"].values,
        "NDMI": landsat_df["NDMI"].values,
        "MNDWI": landsat_df["MNDWI"].values,
        "pet": terraclimate_df["pet"].values,
        "Latitude": template_df["Latitude"].values,
        "Longitude": template_df["Longitude"].values,
        "Sample Date": pd.to_datetime(template_df["Sample Date"], errors="coerce"),
    })
    if use_extra_raw:
        for col in ["NDVI", "swir16", "nir08", "nir", "red", "green", "blue"]:
            if col in landsat_df.columns:
                X[col] = landsat_df[col].values
        for col in ["ppt", "tmax", "tmin", "aet", "def", "soil", "vpd"]:
            if col in terraclimate_df.columns:
                X[col] = terraclimate_df[col].values
    X = add_time_features(X, "Sample Date")
    X = add_spatial_poly(X)
    X = add_interactions(X)
    if use_spatial_grid_mean:
        X = add_spatial_means_same_date(X, date_col="Sample Date", cols=("swir22", "NDMI", "MNDWI", "pet"), grid_deg=0.25)
    if use_lag_rolling:
        X = add_lag_and_rolling(X, group_cols=("Latitude", "Longitude"), date_col="Sample Date",
                                cols=("swir22", "NDMI", "MNDWI", "pet"), lags=(1, 3), rolling_windows=(3, 7))
    X = X.drop(columns=["Sample Date"])
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median(numeric_only=True))
    return X

In [42]:
# Template must have same length as landsat/Terraclimate (wq_data from concat can differ)
template_train = landsat_train_features[["Latitude", "Longitude", "Sample Date"]].copy()
# Normalize Sample Date to days so merge with wq_data matches (wq_data has days from water load)
template_train["Sample Date"] = pd.to_datetime(template_train["Sample Date"], errors="coerce")
template_train["Sample Date"] = (template_train["Sample Date"] - pd.Timestamp("1970-01-01")).dt.days

X_train = build_features(landsat_train_features, Terraclimate_df, template_train,
                         use_extra_raw=True,
                         use_spatial_grid_mean=True,
                         use_lag_rolling=True)

# Align targets to X_train rows via merge (same Lat, Lon, Sample Date)
wq_merge = wq_data[["Latitude", "Longitude", "Sample Date", "Total Alkalinity", "Electrical Conductance", "Dissolved Reactive Phosphorus"]].copy()
if pd.api.types.is_datetime64_any_dtype(wq_merge["Sample Date"]):
    wq_merge["Sample Date"] = (wq_merge["Sample Date"] - pd.Timestamp("1970-01-01")).dt.days
merged = template_train.merge(wq_merge, on=["Latitude", "Longitude", "Sample Date"], how="left")
y_TA = merged["Total Alkalinity"]
y_EC = merged["Electrical Conductance"]
y_DRP = merged["Dissolved Reactive Phosphorus"]

mask = y_TA.notna() & y_EC.notna() & y_DRP.notna() & X_train.notna().all(axis=1)
X = X_train[mask].copy()
y_TA = y_TA[mask]
y_EC = y_EC[mask]
y_DRP = y_DRP[mask]

model_TA, scaler_TA, results_TA = run_pipeline(X, y_TA, "Total Alkalinity")
model_EC, scaler_EC, results_EC = run_pipeline(X, y_EC, "Electrical Conductance")
model_DRP, scaler_DRP, results_DRP = run_pipeline(X, y_DRP, "Dissolved Reactive Phosphorus")


Training Model for Total Alkalinity

Train Evaluation:
R²: 0.863
RMSE: 29.597

Test Evaluation:
R²: -0.050
RMSE: 84.292

Training Model for Electrical Conductance

Train Evaluation:
R²: 0.828
RMSE: 209.734

Test Evaluation:
R²: -0.552
RMSE: 417.673

Training Model for Dissolved Reactive Phosphorus

Train Evaluation:
R²: 0.822
RMSE: 0.231

Test Evaluation:
R²: -0.069
RMSE: 0.720


### Model Performance Summary

<p align="justify">After training and evaluating the models for each water quality parameter, the individual performance metrics are combined into a single summary table. This table consolidates the R² and RMSE values for both in-sample and out-of-sample evaluations, enabling an easy comparison of model performance across Total Alkalinity, Electrical Conductance, and Dissolved Reactive Phosphorus. Such a summary provides a quick overview of how well each model captures the variability in the respective parameter and highlights any differences in predictive accuracy.</p>

In [43]:
# X from wq_data: drop targets AND Sample Date (datetime); keep only numeric for sklearn
X = wq_data.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus', 'Sample Date'])
X = X.select_dtypes(include=[np.number])

y_TA = wq_data['Total Alkalinity']
y_EC = wq_data['Electrical Conductance']
y_DRP = wq_data['Dissolved Reactive Phosphorus']

mask = y_TA.notna() & y_EC.notna() & y_DRP.notna() & X.notna().all(axis=1)
X = X[mask].copy()
y_TA, y_EC, y_DRP = y_TA[mask], y_EC[mask], y_DRP[mask]

model_TA, scaler_TA, results_TA = run_pipeline(X, y_TA, "Total Alkalinity")
model_EC, scaler_EC, results_EC = run_pipeline(X, y_EC, "Electrical Conductance")
model_DRP, scaler_DRP, results_DRP = run_pipeline(X, y_DRP, "Dissolved Reactive Phosphorus")


Training Model for Total Alkalinity

Train Evaluation:
R²: 0.986
RMSE: 10.926

Test Evaluation:
R²: 0.906
RMSE: 27.292

Training Model for Electrical Conductance

Train Evaluation:
R²: 0.994
RMSE: 89.896

Test Evaluation:
R²: 0.957
RMSE: 236.682

Training Model for Dissolved Reactive Phosphorus

Train Evaluation:
R²: 0.929
RMSE: 0.201

Test Evaluation:
R²: 0.714
RMSE: 0.326


In [44]:
results_summary = pd.concat([results_TA, results_EC, results_DRP], ignore_index=True)
results_summary

,Parameter,R2_Train,RMSE_Train,R2_Test,RMSE_Test
0,Total Alkalinity,0.986310,10.925539,0.906003,27.292304
1,Electrical Conductance,0.994090,89.895627,0.957034,236.682006
2,Dissolved Reactive Phosphorus,0.929029,0.201368,0.713824,0.325573


## Submission

<p align="justify">Once you are satisfied with your model’s performance, you can proceed to make predictions for unseen data. To do this, use your trained model to estimate the concentrations of the target water quality parameters — Total Alkalinity, Electrical Conductance, and Dissolved Reactive Phosphorus — for a set of test locations provided in the "Submission_template.csv" file. The predicted results can then be uploaded to the challenge platform for evaluation.</p>

In [ ]:
#Reading the coordinates for the submission
test_file = pd.read_csv(os.path.join(DATA_DIR, "submission_template.csv"))
test_file.head()

In [ ]:
# How many repeated observations per location? (lag/rolling need same (Lat,Lon) over time)
tmp = test_file.copy()
tmp["Sample Date"] = pd.to_datetime(tmp["Sample Date"], errors="coerce")
print("Validation: observations per (Latitude, Longitude)")
print(tmp.groupby(["Latitude", "Longitude"]).size().describe())

<p align="justify">
Similarly, participants can use the <b>Landsat</b> and <b>TerraClimate</b> data extraction demonstration notebooks to produce feature CSVs for their <b>validation</b> data. For convenience, we have already computed and saved example validation outputs as <code>landsat_features_val_V3.csv</code> and <code>Terraclimate_val_df_v3.csv</code>. Participants should save their own extracted files in the same format and column schema; doing so will allow this benchmark notebook to load the validation features directly and run smoothly.
</p>


In [ ]:
landsat_val_features = pd.read_csv(os.path.join(DATA_DIR, "landsat_features_validation.csv"))
landsat_val_features.head()

In [ ]:
Terraclimate_val_df = pd.read_csv(os.path.join(DATA_DIR, "terraclimate_features_validation.csv"))
Terraclimate_val_df.head()

In [ ]:
# Build submission features with same pipeline as training (lag/rolling + spatial grid mean)
X_sub = build_features(landsat_val_features, Terraclimate_val_df, test_file,
                      use_extra_raw=True,
                      use_spatial_grid_mean=True,
                      use_lag_rolling=True)

In [ ]:
# Validation feature matrix ready (same columns as X_train)
X_sub.shape

In [ ]:
# Predict with same scaler/model as training
pred_TA_submission = model_TA.predict(scaler_TA.transform(X_sub))
pred_EC_submission = model_EC.predict(scaler_EC.transform(X_sub))
pred_DRP_submission = model_DRP.predict(scaler_DRP.transform(X_sub))

submission_df = pd.DataFrame({
    "Longitude": test_file["Longitude"].values,
    "Latitude": test_file["Latitude"].values,
    "Sample Date": test_file["Sample Date"].values,
    "Total Alkalinity": pred_TA_submission,
    "Electrical Conductance": pred_EC_submission,
    "Dissolved Reactive Phosphorus": pred_DRP_submission,
})

In [ ]:
submission_df.to_csv("submission.csv", index=False)
print(submission_df.head())

In [ ]:
# Submission feature matrix shape (same cols as training)
X_sub.shape

In [ ]:
# Predictions and submission_df computed above (using X_sub)
submission_df.head()

In [ ]:
# submission_df already created above (after predict)

In [ ]:
#Displaying the sample submission dataframe
submission_df.head()

In [ ]:
#Dumping the predictions into a csv file.
submission_df.to_csv("submission.csv",index = False)

### Upload submission file on platform

Upload the submission.csv on the <a href ="https://challenge.ey.com">platform</a> to get score generated on scoreboard.

## Conclusion

<div align ="justify">Now that you have learned a basic approach to model training, it’s time to try your own approach! Feel free to modify any of the functions presented in this notebook. We look forward to seeing your version of the model and the results. Best of luck with the challenge!</div>